In [2]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
import matplotlib.pyplot as plt
import time

# Step 2: Load the Dataset
file_path = 'walmart-sales-dataset-of-45stores.csv'  # Update this path to match your local file
df = pd.read_csv(file_path)
time.sleep(1)
print("Dataset loaded. Shape:", df.shape)

# Step 3: Drop Missing Values
df = df.dropna()
print("Dropped missing rows. New shape:", df.shape)
time.sleep(1)

# Step 4: Feature Engineering — Time Features
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Year'] = df['Date'].dt.year
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
print("Time-based features added.")
time.sleep(1)

# Step 5: Lag Features — Temporal Memory
# Use 'Store' only if 'Dept' is missing
group_cols = ['Store']
if 'Dept' in df.columns:
    group_cols.append('Dept')

# Sort for lag calculation
df = df.sort_values(group_cols + ['Date'])

# Create lag features
df['Lag_1'] = df.groupby(group_cols)['Weekly_Sales'].shift(1)
df['Lag_2'] = df.groupby(group_cols)['Weekly_Sales'].shift(2)
df['Lag_3'] = df.groupby(group_cols)['Weekly_Sales'].shift(3)

# Rolling mean and std
df['RollingMean_4w'] = df.groupby(group_cols)['Weekly_Sales'].transform(lambda x: x.rolling(4).mean())
df['RollingStd_4w'] = df.groupby(group_cols)['Weekly_Sales'].transform(lambda x: x.rolling(4).std())

print("Lag and rolling features added using group:", group_cols)
time.sleep(1)

#  Step 6: Drop rows with NaNs from lag/rolling features
df = df.dropna()
print("Cleaned lagged rows. New shape:", df.shape)
time.sleep(1)

#  Step 7: Encode Categorical Variables
categorical_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Encoded categorical columns:", categorical_cols)
time.sleep(1)

# Step 8: Log Transform the Target Variable
df['Log_Weekly_Sales'] = np.log1p(df['Weekly_Sales'])
target_column = 'Log_Weekly_Sales'
print("Log transformation applied to target.")
time.sleep(1)

# 🧪 Step 9: Per-Store Modeling Loop
store_scores = {}  # Dictionary to store R² per store

for store_id in df['Store'].unique():
    print(f"\n Processing Store {store_id}")
    
    # Filter data for current store
    df_store = df[df['Store'] == store_id].copy()
    
    # Drop target and original sales from features
    X = df_store.drop(columns=['Weekly_Sales', 'Log_Weekly_Sales'])
    
    # Drop datetime columns if any remain
    datetime_cols = X.select_dtypes(include='datetime64').columns.tolist()
    if datetime_cols:
        X = X.drop(columns=datetime_cols)
    
    y = df_store[target_column]
    
    # 📏 Standardize Features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # 🧪 Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    
    #  Tune Alpha with RidgeCV
    alphas = np.logspace(-3, 3, 100)
    ridge_cv = RidgeCV(alphas=alphas, scoring='r2', cv=5)
    ridge_cv.fit(X_train, y_train)
    
    #  Polynomial Features
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    # Train Ridge Regression
    ridge_model = Ridge(alpha=ridge_cv.alpha_)
    ridge_model.fit(X_train_poly, y_train)
    
    #  Predict and Evaluate
    y_pred = ridge_model.predict(X_test_poly)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Store R² score
    store_scores[store_id] = r2
    
    print(f" Store {store_id} R²: {r2:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f}")

#  Step 10: Summary of R² Scores
print("\n R² Scores by Store:")
for store_id, score in store_scores.items():
    print(f"Store {store_id}: R² = {score:.4f}")

✅ Dataset loaded. Shape: (6435, 8)
🧼 Dropped missing rows. New shape: (6435, 8)
🧠 Time-based features added.
🧠 Lag and rolling features added using group: ['Store']
🧼 Cleaned lagged rows. New shape: (6300, 18)
🔠 Encoded categorical columns: []
🔄 Log transformation applied to target.

🏪 Processing Store 1
📊 Store 1 R²: 0.9815 | RMSE: 0.01 | MAE: 0.01

🏪 Processing Store 2
📊 Store 2 R²: 0.7908 | RMSE: 0.03 | MAE: 0.01

🏪 Processing Store 3
📊 Store 3 R²: 0.8806 | RMSE: 0.03 | MAE: 0.01

🏪 Processing Store 4
📊 Store 4 R²: 0.9343 | RMSE: 0.02 | MAE: 0.01

🏪 Processing Store 5
📊 Store 5 R²: 0.9415 | RMSE: 0.02 | MAE: 0.01

🏪 Processing Store 6
📊 Store 6 R²: 0.8789 | RMSE: 0.03 | MAE: 0.01

🏪 Processing Store 7
📊 Store 7 R²: 0.9505 | RMSE: 0.04 | MAE: 0.03

🏪 Processing Store 8
📊 Store 8 R²: 0.8094 | RMSE: 0.04 | MAE: 0.01

🏪 Processing Store 9
📊 Store 9 R²: 0.8575 | RMSE: 0.03 | MAE: 0.01

🏪 Processing Store 10
📊 Store 10 R²: 0.8988 | RMSE: 0.03 | MAE: 0.01

🏪 Processing Store 11
📊 Store 11 